# RF-DETR Weight-Sharing NAS – Google Colab Runner

Notebook điều phối để kéo code từ GitHub về Google Colab (GPU **L4**), cài đặt package `rfdetr_reproduction`, và chạy toàn bộ 3 giai đoạn reproduction với **checkpoint & kết quả đồng bộ trực tiếp với cấu trúc Google Drive hiện có**:

```text
/content/drive/MyDrive/RFDETR_Checkpoints/
├── nas_supernet_cocoinit_full.pth          (checkpoint COCO-init Epoch 20 có sẵn)
├── nas_train_history.csv
└── results_cocoinit/                       (thư mục lưu kết quả & biểu đồ)
    ├── nas/
    ├── diagnostics/
    ├── grid_search/
    ├── deployment/
    └── baselines/
```


## 1. Mount Google Drive & Pull/Clone Code từ GitHub


In [ ]:
import os
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# === CẤU HÌNH REPO GITHUB CỦA BẠN ===
GITHUB_REPO_URL = "https://github.com/<USERNAME>/<TEN_REPO>.git"  # <-- Điền link repo của bạn
BRANCH = "main"
USE_PRIVATE_TOKEN = False  # Đổi thành True nếu repo private (lưu PAT vào Colab Secrets tên GITHUB_TOKEN)

REPO_DIR = Path("/content/RF-DETR-Reproduction")

repo_url = GITHUB_REPO_URL
if USE_PRIVATE_TOKEN:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    repo_url = GITHUB_REPO_URL.replace("https://", f"https://{token}@")

if (REPO_DIR / ".git").exists():
    print(f"Repo đã tồn tại tại {REPO_DIR} -> chạy git pull ({BRANCH})...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {GITHUB_REPO_URL} (branch={BRANCH}) -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Cài đặt thư viện, Package `rfdetr_reproduction` & Chạy Unit Tests


In [ ]:
!pip install -q "rfdetr @ git+https://github.com/roboflow/rf-detr.git@382607f3d7ecbf513b991a8ee9f1a3abe3a340f0"
!pip install -q pycocotools onnx onnxruntime-gpu ultralytics pytest tqdm
!pip install -q -e .

# Chạy unit test kiểm tra validator, search space (111 / 3948 / 3780) và dataset integrity
!pytest -v


## 3. Ánh xạ cấu trúc Google Drive hiện có (`RFDETR_Checkpoints`) vào Project

Cell này tự động:
1. Trỏ `RFDETR_DATA_DIR` về SSD local của Colab (`/content/data`) để đọc ảnh nhanh nhất.
2. Trỏ `RFDETR_CKPT_DIR` và `RFDETR_OUTPUT_DIR` thẳng vào `/content/drive/MyDrive/RFDETR_Checkpoints/`.
3. **Liên kết tự động checkpoint sẵn có** `nas_supernet_cocoinit_full.pth` → `nas/nas_supernet_latest.pth` và tách sẵn `nas/nas_supernet_ema_only.pth` (nhẹ hơn cho Grid Search & Export) để mọi script nhận ngay trọng số Epoch 20 mà không phải train lại từ đầu.
4. **Liên kết tự động các file kết quả cũ** (`nas_validation_results_cocoinit.csv`, `pareto_frontier_cocoinit.csv`, `nas_train_history.csv`) vào thư mục đích để có thể resume mượt mà.


In [ ]:
import os
import shutil
from pathlib import Path
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/RFDETR_Checkpoints")
DRIVE_RESULTS = DRIVE_ROOT / "results_cocoinit"
DRIVE_FULL_CKPT = DRIVE_ROOT / "nas_supernet_cocoinit_full.pth"

# Thư mục runtime chuẩn cho các script trong project
DATA_DIR = Path("/content/data")
CKPT_DIR = DRIVE_ROOT / "project_checkpoints"
OUTPUT_DIR = DRIVE_RESULTS

os.environ["RFDETR_DATA_DIR"] = str(DATA_DIR)
os.environ["RFDETR_CKPT_DIR"] = str(CKPT_DIR)
os.environ["RFDETR_OUTPUT_DIR"] = str(OUTPUT_DIR)

for d in [
    DATA_DIR,
    CKPT_DIR / "nas",
    CKPT_DIR / "pretrained",
    CKPT_DIR / "diagnostics",
    OUTPUT_DIR / "nas",
    OUTPUT_DIR / "diagnostics",
    OUTPUT_DIR / "grid_search",
    OUTPUT_DIR / "deployment",
    OUTPUT_DIR / "baselines",
]:
    d.mkdir(parents=True, exist_ok=True)


def _link_or_copy(src: Path, dst: Path):
    if not src.exists() or dst.exists():
        return
    try:
        os.symlink(src, dst)
        print(f"[Symlink] {dst.name} -> {src}")
    except OSError:
        shutil.copy2(src, dst)
        print(f"[Copy]    {src} -> {dst}")


# 1. Liên kết checkpoint Epoch 20 sẵn có vào chuẩn project
nas_latest = CKPT_DIR / "nas" / "nas_supernet_latest.pth"
nas_ema = CKPT_DIR / "nas" / "nas_supernet_ema_only.pth"

if DRIVE_FULL_CKPT.exists():
    _link_or_copy(DRIVE_FULL_CKPT, nas_latest)
    if not nas_ema.exists():
        print("Đang tách bản nhẹ nas_supernet_ema_only.pth từ nas_supernet_cocoinit_full.pth ...")
        ckpt = torch.load(DRIVE_FULL_CKPT, map_location="cpu", weights_only=False)
        torch.save(
            {
                "epoch": ckpt.get("epoch", 20),
                "ema_model": ckpt.get("ema_model", ckpt.get("model")),
                "ema_updates": ckpt.get("ema_updates", 0),
                "provenance": ckpt.get("provenance"),
            },
            nas_ema,
        )
        del ckpt
        print(f"[Created] {nas_ema} ({nas_ema.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"[INFO] Không tìm thấy {DRIVE_FULL_CKPT} -> sẽ train mới khi chạy Stage 1.")

# 2. Liên kết các file kết quả đã có trên Drive để resume Grid Search / Deployment
_link_or_copy(DRIVE_RESULTS / "nas_validation_results_cocoinit.csv", OUTPUT_DIR / "grid_search" / "nas_validation_results.csv")
_link_or_copy(DRIVE_RESULTS / "pareto_frontier_cocoinit.csv", OUTPUT_DIR / "grid_search" / "pareto_frontier.csv")
_link_or_copy(DRIVE_RESULTS / "rfdetr_nas_supernet" / "nas_train_history.csv", OUTPUT_DIR / "nas" / "nas_train_history.csv")

print("\n=== TRẠNG THÁI ĐƯỜNG DẪN ===")
print("RFDETR_DATA_DIR  =", os.environ["RFDETR_DATA_DIR"])
print("RFDETR_CKPT_DIR  =", os.environ["RFDETR_CKPT_DIR"])
print("RFDETR_OUTPUT_DIR=", os.environ["RFDETR_OUTPUT_DIR"])
print("Latest ckpt      =", nas_latest.exists(), f"({nas_latest})")
print("EMA-only ckpt    =", nas_ema.exists(), f"({nas_ema})")


## 4. Giai đoạn 1 – Kiểm tra môi trường, Chuẩn bị VOC2007 & Huấn luyện NAS


In [ ]:
# 4.1. Ghi provenance (GPU L4, CUDA, PyTorch, Git SHA) -> results_cocoinit/provenance.json
!python scripts/setup_environment.py --require-cuda

# 4.2. Tải VOC2007 & chuyển sang COCO JSON (lọc difficult=1, kiểm tra zero-overlap)
!python scripts/prepare_dataset.py


In [ ]:
# 4.3. Huấn luyện Weight-Sharing NAS Supernet
# Mặc định đặt False vì bạn đã có sẵn nas_supernet_cocoinit_full.pth (Epoch 20) trên Drive.
# Đổi thành True nếu muốn train tiếp (vd: NAS_EPOCHS = 25) hoặc train lại từ đầu (thêm --no-resume).
RUN_TRAIN_NAS = False
NAS_EPOCHS = 20

if RUN_TRAIN_NAS:
    !python scripts/train_nas.py --epochs {NAS_EPOCHS}
    # Đồng bộ lại tên file gốc trên Drive để giữ nhất quán cấu trúc cũ
    if not DRIVE_FULL_CKPT.is_symlink() and (CKPT_DIR / "nas" / "nas_supernet_latest.pth").exists():
        shutil.copy2(CKPT_DIR / "nas" / "nas_supernet_latest.pth", DRIVE_FULL_CKPT)
    hist_src = OUTPUT_DIR / "nas" / "nas_train_history.csv"
    if hist_src.exists():
        shutil.copy2(hist_src, DRIVE_ROOT / "nas_train_history.csv")
        (DRIVE_RESULTS / "rfdetr_nas_supernet").mkdir(parents=True, exist_ok=True)
        shutil.copy2(hist_src, DRIVE_RESULTS / "rfdetr_nas_supernet" / "nas_train_history.csv")
else:
    print("Bỏ qua train_nas.py (đang dùng sẵn checkpoint Epoch 20 trên Drive).")


## 5. Giai đoạn 2 – Thí nghiệm Chẩn đoán (Gradient Conflict, Hypo 1, Hypo 2) & Grid Search


In [ ]:
# 5a. Gradient Conflict Diagnosis (Mục 5b)
from IPython.display import Image, display
import pandas as pd

RUN_GRADIENT_CONFLICT = True
GC_TAG = "ep20"

if RUN_GRADIENT_CONFLICT:
    !python scripts/run_gradient_conflict.py --ckpt "{DRIVE_FULL_CKPT}" --tag {GC_TAG}
    img_gc = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_heatmap_{GC_TAG}.png"
    csv_gc = OUTPUT_DIR / "diagnostics" / f"gradient_conflict_summary_{GC_TAG}.csv"
    if csv_gc.exists():
        display(pd.read_csv(csv_gc).round(4))
    if img_gc.exists():
        display(Image(filename=str(img_gc)))


In [ ]:
# 5b. Hypothesis 1 – Tốc độ hội tụ riêng của từng Architecture (Mục 5d)
# Tự động lưu & resume trạng thái qua project_checkpoints/diagnostics/hypo1_state.pth trên Drive
RUN_HYPO1 = True
HYPO1_EPOCHS = 20
HYPO1_EVAL_EVERY = 2

if RUN_HYPO1:
    !python scripts/run_hypothesis1.py --epochs {HYPO1_EPOCHS} --eval-every {HYPO1_EVAL_EVERY}
    csv_h1 = OUTPUT_DIR / "diagnostics" / "hypo1_t90_summary.csv"
    img_h1 = OUTPUT_DIR / "diagnostics" / "hypo1_convergence.png"
    if csv_h1.exists():
        display(pd.read_csv(csv_h1).round(4))
    if img_h1.exists():
        display(Image(filename=str(img_h1)))


In [ ]:
# 5c. Hypothesis 2 – Weight Drift / Interference trên mạng nhỏ khi tăng p(A) (Mục 5e)
# Nạp trực tiếp từ nas_supernet_cocoinit_full.pth (không ghi đè checkpoint gốc)
RUN_HYPO2 = True
HYPO2_EXTRA_EPOCHS = 5

if RUN_HYPO2:
    !python scripts/run_hypothesis2.py --ckpt "{DRIVE_FULL_CKPT}" --extra-epochs {HYPO2_EXTRA_EPOCHS}
    csv_h2 = OUTPUT_DIR / "diagnostics" / "hypo2_summary.csv"
    img_h2 = OUTPUT_DIR / "diagnostics" / "hypo2_interference.png"
    if csv_h2.exists():
        display(pd.read_csv(csv_h2).round(4))
    if img_h2.exists():
        display(Image(filename=str(img_h2)))


In [ ]:
# 5d. Post-Training Grid Search & Pareto Frontier (Mục 6)
# Tự động resume từ nas_validation_results_cocoinit.csv nếu đã có sẵn trên Drive
RUN_GRID_SEARCH = False
GRID_MODE = "representative"  # "representative" | "full"

if RUN_GRID_SEARCH:
    !python scripts/run_grid_search.py --mode {GRID_MODE} --ckpt "{CKPT_DIR / 'nas' / 'nas_supernet_ema_only.pth'}"
    pareto_csv = OUTPUT_DIR / "grid_search" / "pareto_frontier.csv"
    pareto_png = OUTPUT_DIR / "grid_search" / "nas_pareto_frontier.png"
    if pareto_csv.exists():
        display(pd.read_csv(pareto_csv))
    if pareto_png.exists():
        display(Image(filename=str(pareto_png)))
else:
    print("Bỏ qua Grid Search (đã có sẵn pareto_frontier_cocoinit.csv trên Drive).")


## 6. Giai đoạn 3 – Deployment (ONNX/TensorRT/Throttling), YOLO Baselines & Segmentation


In [ ]:
RUN_DEPLOYMENT = False
RUN_BASELINES = False
RUN_SEGMENTATION = False

EMA_CKPT = CKPT_DIR / "nas" / "nas_supernet_ema_only.pth"
PARETO_CSV = OUTPUT_DIR / "grid_search" / "pareto_frontier.csv"

if RUN_DEPLOYMENT:
    !python scripts/run_deployment.py --ckpt "{EMA_CKPT}" --pareto-csv "{PARETO_CSV}"
    thr_png = OUTPUT_DIR / "deployment" / "throttling_comparison.png"
    if thr_png.exists():
        display(Image(filename=str(thr_png)))

if RUN_BASELINES:
    !python scripts/run_baselines.py --ckpt "{EMA_CKPT}" --pareto-csv "{PARETO_CSV}" --epochs 20
    comp_csv = OUTPUT_DIR / "baselines" / "fair_baseline_comparison_test.csv"
    if comp_csv.exists():
        display(pd.read_csv(comp_csv))

if RUN_SEGMENTATION:
    !python scripts/run_segmentation.py --enable


## 7. Đồng bộ tổng hợp về đúng tên file trên `RFDETR_Checkpoints/results_cocoinit/`


In [ ]:
import shutil

sync_pairs = [
    (OUTPUT_DIR / "grid_search" / "nas_validation_results.csv", DRIVE_RESULTS / "nas_validation_results_cocoinit.csv"),
    (OUTPUT_DIR / "grid_search" / "pareto_frontier.csv", DRIVE_RESULTS / "pareto_frontier_cocoinit.csv"),
    (OUTPUT_DIR / "grid_search" / "nas_pareto_frontier.png", DRIVE_RESULTS / "nas_pareto_frontier.png"),
    (OUTPUT_DIR / "deployment" / "throttling_comparison.png", DRIVE_RESULTS / "throttling_comparison.png"),
    (OUTPUT_DIR / "baselines" / "fair_baseline_comparison_test.csv", DRIVE_RESULTS / "fair_baseline_comparison_test.csv"),
]

for src, dst in sync_pairs:
    if src.exists() and not src.is_symlink() and src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
        print(f"[Synced] {src.name} -> {dst}")

print("\n✅ Danh sách file hiện có trong results_cocoinit/:")
for p in sorted(DRIVE_RESULTS.rglob("*")):
    if p.is_file():
        print("  -", p.relative_to(DRIVE_RESULTS))
